In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [ ]:
spark = SparkSession.builder \
      .appName("GraphFeature") \
      .getOrCreate()

In [ ]:
path = '/content/drive/MyDrive/Mining of Massive Dataset/Processed_Data'

edge_df = spark.read.parquet(f"{path}/edge_table")
node_df = spark.read.parquet(f"{path}/node_table")
features_df = spark.read.parquet(f"{path}/node_features")

print(f"Số lượng giao dịch: {edge_df.count()}")

Số lượng giao dịch: 31898238


In [ ]:
print(f"Số lượng nút (node): {node_df.count()}")
print(f"Số lượng đặc trưng (feature): {features_df.count()}")

Số lượng nút (node): 2077023
Số lượng đặc trưng (feature): 2077023


In [ ]:
edge_df.printSchema()
edge_df.show(5)

root
 |-- src: string (nullable = true)
 |-- dst: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- payment_format: string (nullable = true)
 |-- is_laundering_tx: integer (nullable = true)

+----------------+----------------+------------------+-------------------+--------------+----------------+
|             src|             dst|            amount|          timestamp|payment_format|is_laundering_tx|
+----------------+----------------+------------------+-------------------+--------------+----------------+
|316683_833FE23B0|316683_833FE23B0|2841.3720000000003|2022-09-01 00:06:00|  Reinvestment|               0|
|382180_8510F1EB0|382180_8510F1EB0|377.11440000000005|2022-09-01 00:27:00|  Reinvestment|               0|
| 89090_836F88990| 89090_836F88990|368.38800000000003|2022-09-01 00:25:00|  Reinvestment|               0|
|160474_8194BAEE0|160474_8194BAEE0| 7.497599999999999|2022-09-01 00:14:00|  Reinvestment|               0

In [ ]:
node_df.printSchema()
node_df.show(5)

root
 |-- account_id: string (nullable = true)
 |-- bank_id: integer (nullable = true)
 |-- entity_id: string (nullable = true)
 |-- label: integer (nullable = true)

+-----------+-------+-----------+-----+
| account_id|bank_id|  entity_id|label|
+-----------+-------+-----------+-----+
|0_800047930|      0|2AA1CA62650|    0|
|0_80006C140|      0|2AA1CA8DB10|    0|
|0_80006F150|      0|2AA1CA71020|    0|
|0_8000719B0|      0|2AA1CA62710|    0|
|0_800071D50|      0|2AA1CA62770|    0|
+-----------+-------+-----------+-----+
only showing top 5 rows


In [ ]:
features_df.printSchema()
features_df.show(5)

root
 |-- account_id: string (nullable = true)
 |-- in_degree: long (nullable = true)
 |-- out_degree: long (nullable = true)
 |-- unique_senders: long (nullable = true)
 |-- unique_receivers: long (nullable = true)
 |-- total_sent: double (nullable = true)
 |-- total_received: double (nullable = true)
 |-- avg_tx_amount: double (nullable = true)
 |-- avg_tx_per_day: double (nullable = true)
 |-- avg_time_gap: double (nullable = true)

+-----------+---------+----------+--------------+----------------+-------------------+--------------------+--------------------+--------------+------------------+
| account_id|in_degree|out_degree|unique_senders|unique_receivers|         total_sent|      total_received|       avg_tx_amount|avg_tx_per_day|      avg_time_gap|
+-----------+---------+----------+--------------+----------------+-------------------+--------------------+--------------------+--------------+------------------+
|0_800105DB0|       33|       212|             5|              16|  544

In [ ]:
num_partitions = 200

edge_df = edge_df.repartition(num_partitions)
node_df = node_df.repartition(num_partitions)
features_df = features_df.repartition(num_partitions)

In [ ]:
from pyspark.sql.functions import col

# Lấy tập hợp các account_id duy nhất từ node_df
node_ids = node_df.select("account_id").distinct()

# Kiểm tra xem src trong edge_df có nằm trong node_df không
src_in_nodes = edge_df.select("src").distinct().join(node_ids, edge_df.src == node_ids.account_id, "inner")

# Kiểm tra xem dst trong edge_df có nằm trong node_df không
dst_in_nodes = edge_df.select("dst").distinct().join(node_ids, edge_df.dst == node_ids.account_id, "inner")

print(f"Số lượng src duy nhất: {edge_df.select('src').distinct().count()}")
print(f"Số lượng src khớp với node_df: {src_in_nodes.count()}")
print(f"Số lượng dst duy nhất: {edge_df.select('dst').distinct().count()}")
print(f"Số lượng dst khớp với node_df: {dst_in_nodes.count()}")

Số lượng src duy nhất: 2013639
Số lượng src khớp với node_df: 2013639
Số lượng dst duy nhất: 1689937
Số lượng dst khớp với node_df: 1689937


In [ ]:
csv_base_path = '/content/drive/MyDrive/Mining of Massive Dataset/Processed_Data'

# Save edge_df to CSV using Spark's native writer
# This creates a directory with multiple part-xxxxx.csv files
edge_df.write.mode("overwrite").csv(f"{csv_base_path}/edge_df_csv", header=True)
print("edge_df saved successfully to edge_df_csv directory.")

# Save node_df to CSV using Spark's native writer
node_df.write.mode("overwrite").csv(f"{csv_base_path}/node_df_csv", header=True)
print("node_df saved successfully to node_df_csv directory.")

# Save features_df to CSV using Spark's native writer
features_df.write.mode("overwrite").csv(f"{csv_base_path}/features_df_csv", header=True)
print("features_df saved successfully to features_df_csv directory.")

edge_df saved successfully to edge_df_csv directory.
node_df saved successfully to node_df_csv directory.
features_df saved successfully to features_df_csv directory.
